In [ ]:
import mlflow
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, recall_score, f1_score

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare features for training:
    - drop Name, Ticket, Fare, and Cabin
    - fill missing cells of Age and Embarke
    - combine SibSp and Parch into 1 column FamilySize
    - split age to 5 groups: Infant, child, Teenager, Adult, and Senior (0-4)
    - encode categorical columns: Sex, Embarked, PClass
    - add 2 significant indicators: is_infant_or_female, is_family_size_2_to_4_or_upper_class
    """
    
    # Fill missing ages with the median of people with the same PClass and Sex
    # Count NaN values in 'Age' before filling
    # nan_count_before = df['Age'].isna().sum()

    # Perform the fill operation
    # df = df.fillna({'Age': df.groupby(['Pclass', 'Sex'])['Age'].transform('median')})

    # The number of filled values is equal to nan_count_before
    # print(f"Number of Age values filled: {nan_count_before}")
    # Fill missing Embarked with the most frequent port of people with the same PClass and Sex
    # df = df.fillna({'Embarked': df.groupby(['Pclass', 'Sex'])['Embarked'].transform(lambda x: x.mode()[0])})
    
    df[['Title', 'LastName']] = df['Name'].apply(lambda x: pd.Series(extract_title_and_last_name(x))).astype('category')
    
    df = df.drop(['PassengerId', 'Name', 'LastName', 'Ticket', 'Cabin'], axis=1)
    
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df = df.drop(['SibSp', 'Parch'], axis=1)
    
    df['Age'] = pd.cut(df['Age'], bins=[0, 6, 12, 35, 65, 150], labels=[0, 1, 2, 3, 4])
    # df['Fare'] = pd.cut(df['Fare'], bins=[0, 20, 75, 1000], labels=[0, 1, 2])
    
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Embarked'] = df['Embarked'].map({'C': 0, 'S': 1, 'Q': 2}).astype('category')
    df['Pclass'] = df['Pclass'] - 1
    
    print(df.dtypes)

    return df.apply(pd.to_numeric, errors='coerce')

df = pd.read_csv('../data/raw/train.csv')
df = build_features(df)

target_col = 'Survived'

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)
 
model = XGBClassifier(
        n_estimators=300,
        learning_rate=0.11,
        max_depth=4,
        random_state=42,
        n_jobs=-1,
        enable_categorical=True,
        early_stopping_rounds=10,
        eval_metric="logloss"
)
    
with mlflow.start_run():
    # Define K-Fold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    # Lists to store metrics for each fold
    accuracies = []
    recalls = []
    f1s = []
    
    # Loop through each fold
    for fold, (train_index, test_index) in enumerate(kf.split(X)):
        X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
        y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]
        
        eval_set = [(X_train_fold, y_train_fold), (X_test_fold, y_test_fold)]
        
        # Train model
        model.fit(X_train_fold, y_train_fold, eval_set=eval_set)
        preds = model.predict(X_test_fold)
        
        # Calculate metrics
        acc = accuracy_score(y_test_fold, preds)
        rec = recall_score(y_test_fold, preds)
        f1 = f1_score(y_test_fold, preds)
        
        # Store metrics
        accuracies.append(acc)
        recalls.append(rec)
        f1s.append(f1)
        
        # Log metrics for this fold
        mlflow.log_metric(f"accuracy_fold", acc, step=fold)
        mlflow.log_metric(f"recall_fold", rec, step=fold)
    
    # Log average and standard deviation
    mlflow.log_metric("accuracy_mean", np.mean(accuracies))
    mlflow.log_metric("accuracy_std", np.std(accuracies))
    mlflow.log_metric("recall_mean", np.mean(recalls))
    mlflow.log_metric("recall_std", np.std(recalls))
    
    # Log parameters and dataset
    mlflow.log_param("n_estimators", 300)
    train_ds = mlflow.data.from_pandas(df, source="training_data")
    mlflow.log_input(train_ds, context="training")
    
    print(f"K-Fold Complete. Avg F1 Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}. Avg Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}, Avg Recall: {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
    
test_df = pd.read_csv('../data/raw/test.csv')
passengerId = test_df['PassengerId']
test_df = build_features(test_df)
preds = model.predict(test_df)
submission = pd.DataFrame({
    'PassengerId': passengerId,
    'Survived': preds
})
submission.head()
submission.to_csv('../data/raw/gender_submission.csv', index=False)

In [ ]:
import mlflow
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, recall_score, f1_score

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare features for training:
    - drop Name, Ticket, Fare, and Cabin
    - fill missing cells of Age and Embarke
    - combine SibSp and Parch into 1 column FamilySize
    - split age to 5 groups: Infant, child, Teenager, Adult, and Senior (0-4)
    - encode categorical columns: Sex, Embarked, PClass
    - add 2 significant indicators: is_infant_or_female, is_family_size_2_to_4_or_upper_class
    """
    df = df.drop(['PassengerId', 'Name', 'Ticket', 'Fare', 'Cabin'], axis=1)
    
    # Fill missing ages with the median of people with the same PClass and Sex
    # Count NaN values in 'Age' before filling
    nan_count_before = df['Age'].isna().sum()

    # Perform the fill operation
    df = df.fillna({'Age': df.groupby(['Pclass', 'Sex'])['Age'].transform('median')})

    # The number of filled values is equal to nan_count_before
    print(f"Number of Age values filled: {nan_count_before}")
    # Fill missing Embarked with the most frequent port of people with the same PClass and Sex
    df = df.fillna({'Embarked': df.groupby(['Pclass', 'Sex'])['Embarked'].transform(lambda x: x.mode()[0])})
    
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df = df.drop(['SibSp', 'Parch'], axis=1)
    
    df['Age'] = pd.cut(df['Age'], bins=[0, 6, 12, 25, 35, 65, 150], labels=[0, 1, 2, 3, 4, 5])
    
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Embarked'] = df['Embarked'].map({'C': 0, 'S': 1, 'Q': 2})
    df['Pclass'] = df['Pclass'] - 1
    
    # df['is_infant_or_female'] = ((df['Age'] == 0) | (df['Sex'] == 1)).astype('int')
    # df['is_family_size_average'] = ((df['FamilySize'] >= 2) & (df['FamilySize'] <= 4)).astype('category')
    # df['is_upper_and_from_cherbourg'] = ((df['Embarked'] == 0) & (df['Pclass'] == 2))
    
    # df.drop(['Sex', 'Pclass'], axis=1)
    print(df.columns)

    return df.apply(pd.to_numeric, errors='coerce')

df = pd.read_csv('../data/raw/train.csv')
df = build_features(df)

target_col = 'Survived'

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)
 
model = XGBClassifier(
        n_estimators=300,
        learning_rate=0.11,
        max_depth=4,
        random_state=42,
        n_jobs=-1,
        enable_categorical=True,
        early_stopping_rounds=10,
        eval_metric="logloss"
)
    
with mlflow.start_run():
    # Define K-Fold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    # Lists to store metrics for each fold
    accuracies = []
    recalls = []
    f1s = []
    
    # Loop through each fold
    for fold, (train_index, test_index) in enumerate(kf.split(X)):
        X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
        y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]
        
        eval_set = [(X_train_fold, y_train_fold), (X_test_fold, y_test_fold)]
        
        # Train model
        model.fit(X_train_fold, y_train_fold, eval_set=eval_set)
        preds = model.predict(X_test_fold)
        
        # Calculate metrics
        acc = accuracy_score(y_test_fold, preds)
        rec = recall_score(y_test_fold, preds)
        f1 = f1_score(y_test_fold, preds)
        
        # Store metrics
        accuracies.append(acc)
        recalls.append(rec)
        f1s.append(f1)
        
        # Log metrics for this fold
        mlflow.log_metric(f"accuracy_fold", acc, step=fold)
        mlflow.log_metric(f"recall_fold", rec, step=fold)
    
    # Log average and standard deviation
    mlflow.log_metric("accuracy_mean", np.mean(accuracies))
    mlflow.log_metric("accuracy_std", np.std(accuracies))
    mlflow.log_metric("recall_mean", np.mean(recalls))
    mlflow.log_metric("recall_std", np.std(recalls))
    
    # Log parameters and dataset
    mlflow.log_param("n_estimators", 300)
    train_ds = mlflow.data.from_pandas(df, source="training_data")
    mlflow.log_input(train_ds, context="training")

    print(f"K-Fold Complete. Avg F1 Score: {np.mean(f1):.4f} ± {np.std(f1s):.4f}. Avg Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}, Avg Recall: {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
    
test_df = pd.read_csv('../data/raw/test.csv')
passengerId = test_df['PassengerId']
test_df = build_features(test_df)
preds = model.predict(test_df)
submission = pd.DataFrame({
    'PassengerId': passengerId,
    'Survived': preds
})
submission.head()
submission.to_csv('../data/raw/gender_submission.csv', index=False)